# OOP Part 2

## Encapsulation; Copying Objects; Dynamic Memory Allocation of Objects

<br><br><br><br><br>
<br><br><br><br><br>

# 1. Encapsulation

### * In the last Mario example, the *client* code never makes reference to the attribute variables `m.name_`, `m.coins_`, and `m.lives_`.  

### * Instead, the creation, accessing and changing of these values for an object are done via **methods**.  

### * This is desirable!  As the designer of a class, your goal is to design an interface (that is, a set of methods) that is strong enough to capture all the things a client might want to do with objects. To put it starkly:

    Code inside a class *definition* can reference attribute variables.
    
    *Client* code should try to NOT reference attribute variables -- it ought to only interact with ENTIRE objects, via INTERFACE METHODS.

### * This idea is known as *encapsulation*.   The examples with methods are "better", because they don't violate encapsulation.  

<br><br><br><br><br>
<br><br><br><br><br>

### * Why is this important?  When you BUILD a car, the technical bits under the hood are important, but when you DRIVE the car, the technical bits are distracting at best, harmful at worst.

### * The attribute variables are frequently thought of as technical details used to represent the object.  The user isn't meant to directly look at or modify them; instead, they ought to interact with them through the interface -- that is, the methods -- which (if well-designed) are guaranteed to access or modify objects in appropriate ways.

### * In fact, that's what the underscores signify.   They are a custom, meant to convey to other programmers who are using the class: "hey, I am a technical detail! Don't reference me directly! If you *do* use me you'll probably screw something up, and it's your own fault then!  Be smart and use the interface instead!"

<br><br><br><br><br>
<br><br><br><br><br>


### * When client code *uses* the class, it usually won't refer directly to attribute variables of objects, but rather it will instead use methods to change/access those objects. 


### * I.e. should do less of this:

`x.attribute_ = value`

### * and more of this: 

`x.set_attribute(value)`

### * "The attribute variables are the technical underside, while the methods are the usable interface" -- that's what those underscores signify.

### * Ok, but *why is that important*? Is it just a matter of taste? NO! Two reasons. But first, a quick exercise.

<br><br><br><br><br>
<br><br><br><br><br>

<br><br><br><br><br>
<br><br><br><br><br>

In [ ]:
////employeeEncapsulation.cpp
// Example 1a: Employee without encapsulation
// YOU TRY: Rewrite this so the class has a giveRaise() method (takes a percentage),
// and the client code uses that method instead of directly modifying salary_.
#include<iostream>
#include<string>
class Employee{
    public:
    std::string name_; std::string role_; double salary_;
    Employee(std::string n, std::string r){
        this->name_ = n;
        this->role_ = r;
        this->salary_ = 10000;
    }
    // TODO: write a giveRaise() method that takes a percentage
    // and increases salary_ by that percent



};
int main(){
    Employee e("Areeba", "Math Lecturer");
    // This directly modifies the attribute -- not great!
    e.salary_ *= 1.15;
    std::cout << e.name_ << " now earns " << e.salary_ << "\n";
    // TODO: rewrite the line above to use your giveRaise() method instead


}

<br><br><br><br><br>

### *A note on `const` methods:* Notice the `const` after `display()`'s signature. This tells the compiler that `display()` **will not modify the object** — "I'm only looking, not touching." Good practice for accessor methods. We'll revisit `const` later.

<br><br><br><br><br>

### **Challenge (if you finish early):** Change `salary_` from a `double` in dollars to an `int` in cents (e.g., 10000 dollars → 1000000 cents). Update `giveRaise()` and `display()` so they still work — `display()` should still print the salary in dollars. Does anything in `main()` need to change?

<br><br><br><br><br>

<br><br><br><br><br>
<br><br><br><br><br>

### Now, let's look at two reasons why encapsulation is important.

### **1.** (The less important reason) *Interacting with objects via methods ensures that clients interact with objects CORRECTLY.*

In [ ]:
//// badClientCode.cpp
// Example 1b: Drawing cards, badly
//% CXXFLAGS -std=c++11 
#include <iostream>
#include <vector>
#include <cstdlib> // for random number generation via rand() -- although this isn't the best way to do it
#include <ctime> // setting the seed for rand so that different values are generated
#include "areebas_card_v1.cpp" // contains the definitions for Card and Deck - remember, really would want .h file instead

int main() {
    srand(static_cast<unsigned int>(time(0))); // Seed for random number generation
    Deck d; // Create a deck.

    // Pick 10 cards, in a flagrantly wrong way.
    for (int i = 0; i < 10; ++i) {
        int card_num = rand() % d._cards.size(); // Look at the length of the deck, and pick an index in that range
        Card current_card = d._cards[card_num]; // Pick the card in that index
        current_card.display(); // Display it
    }

    return 0;
}

### * The problem here is that none of the code in the first loop actually takes care of the part where cards are *removed* from the deck after they are picked.  

### * Maybe that's a dumb mistake -- but the more complex your classes get, the more difficult it gets to avoid similar mistakes.  

### * Sticking to well-designed methods is a way to avoid such mistakes.

<br><br><br><br><br>
<br><br><br><br><br>


### **2.** (The more important reason) *Interacting with objects via methods ensures that your client code won't be broken by IMPLEMENTATION CHANGES.*

### * Imagine that, after I had already started using my class, I decided to change how I represented a `Deck`. Instead of using a list of `Card`s which we shuffle at the time of initialization, suppose that I decided that it was "more efficient" to represent a `Deck` as a 4 x 13 vector, containing boolean values: the rows will correspond to all the possible suits, the columns will correspond to all the possible ranks, and the values will be `True` if the corresponding card has already been drawn, and `False` if it has not been drawn.  Whatever.

### * The lines of code that **only reference methods** will still work if we use the updated class; the code that references attributes will break!

### * As a simpler example: remember the `Employee` class? What if we decided to store salary as an `int` in *cents* instead of a `double` in dollars, to avoid floating-point precision issues? Client code that does `e.salary_ *= 1.15` would break -- `salary_` doesn't exist anymore! But `e.giveRaise(15)` would still work perfectly, because the method handles the internal details.

In [ ]:
////moreClientcode.cpp
// Example 1c: Changing Implementation
//% CXXFLAGS -std=c++11 

#include <iostream>
#include <vector>
#include <cstdlib> // included for using rand()
#include <ctime>   // included for setting the seed for rand()
#include "areebas_card_v2.cpp" // contains the definitions for Card and Deck

int main() {
    Deck d; // Create a deck.

    std::cout << "The good loop:\n" << std::endl;
    for (int i = 0; i < 10; ++i) { // This only uses methods.
        Card c = d.draw();    // It will NOT be broken by a change in implementation.
        c.display();          // 
    }

    std::cout << "\nThe bad loop:\n" << std::endl; 
    srand(static_cast<unsigned int>(time(0))); 
    for (int i = 0; i < 10; ++i) {
        int card_num = rand() % d._cards.size(); // This references the member variable ._cards
                                                 // and it WILL be broken after a change in implementation.
        d._cards[card_num].display(); // Display it
        d._cards.erase(d._cards.begin() + card_num); // Remove the card from the deck 
    }

    return 0;
}

### * So, as a client, respecting the so-called "privacy" of attribute variables means that your code will be much less likely to break in the event of implementation change! 

### * And as a class designer, you can *refactor* your class safely so that it works better or more efficiently, as long as you don't change the way the interface works.

<br><br><br><br><br>
<br><br><br><br><br>
<br><br><br><br><br>
<br><br><br><br><br>

# 2. Copying Objects

### * Last class, some of you wrote code like this to track the highest card:

```cpp
Card highest = card1;
if (card2.beats(highest)) highest = card2;
if (card3.beats(highest)) highest = card3;
highest.display();
```

### * This works! But what exactly is happening when we write `Card highest = card1;`? Is `highest` the same object as `card1`, or a separate copy?

### * Let's investigate using the `Character` class.



### * Before we investigate objects, let's recall how copying works with types we already know.

<br><br><br><br><br>

In [ ]:
////copyingReview.cpp
// Example 2a: Copying with primitives vs. pointers
#include<iostream>
int main(){
    // Case 1: Copying a primitive
    int x = 5;
    int y = x;   // what does y get?
    y = 99;
    std::cout << "x = " << x << ", y = " << y << "\n";

    // Case 2: Copying a pointer
    int* p = new int(5);
    int* q = p;   // what does q get?
    *q = 99;
    std::cout << "*p = " << *p << ", *q = " << *q << "\n";
    std::cout << "p = " << p << ", q = " << q << "\n";
    delete p;
    // what else should we remember to do?
}

<br><br><br><br><br>

### * **The rule:** copying copies the variable itself. For an `int`, that's the value — two independent copies. For a pointer, it's the address — two pointers to the **same** heap data.

### * So what about objects of our own classes?

<br><br><br><br><br>

In [ ]:
////copyingObjects.cpp
// Example 2b: Copying objects -- are they the same object?
#include<iostream>
#include<string>
class Character{
   public: std::string name_; int lives_, coins_;
   Character(std::string n){
        this->name_ = n; this->lives_ = 3; this->coins_ = 0;
   }
   void display() const {
        std::cout << this->name_ << ": " << this->lives_ << " lives, " << this->coins_ << " coins\n";
   }
   void collect_coins(int num_coins){
       this->coins_ += num_coins;
   }
};
int main(){
    Character m("Mario");
    Character m2 = m;
    std::cout << (&m == &m2) << "\n";
    std::cout << &m << " " << &m2 << "\n";
    m2.collect_coins(20);
    m.display();
    m2.display();
}

<br><br><br><br><br><br><br><br><br><br>

### * They are at **different addresses** -- `m2` is a completely independent object! Modifying `m2` does not affect `m`.

### * When we write `Character m2 = m;`, the compiler calls the **copy constructor**. Since we didn't write one, the compiler generates a **default copy constructor** that copies each member variable one by one: `m2.name_` gets a copy of `m.name_`, `m2.lives_` gets a copy of `m.lives_`, etc.



### * We can write our own copy constructor to see exactly when copying happens. Don't worry about the syntax of the parameter for now — we'll explain it shortly.

In [ ]:
////copyConstructor.cpp
// Example 2c: Writing our own copy constructor
#include<iostream>
#include<string>
class Character{
   public: std::string name_; int lives_, coins_;
   Character(std::string n){
        this->name_ = n; this->lives_ = 3; this->coins_ = 0;
        std::cout << "Constructor called for " << n << "\n";
   }
   // Copy constructor -- takes a const reference to another object of the same class
   Character(const Character& other){
        // Print a message and copy each member from other




   }
   void display() const {
        std::cout << this->name_ << ": " << this->lives_ << " lives, " << this->coins_ << " coins\n";
   }
   void collect_coins(int num_coins){
       this->coins_ += num_coins;
   }
};
int main(){
    Character m("Mario");
    m.collect_coins(50);

    // Two ways to call the copy constructor:



}

<br><br><br><br><br><br><br><br><br><br>

### *A note on `const Character& other`:* We haven't covered `const` or `&` (references) yet — we will later. For now: `&` means "don't copy `other`, just look at the original directly," and `const` means "promise not to change it." Why? If the copy constructor had to *copy* its argument... that would call the copy constructor... infinite loop!

<br><br><br><br><br><br><br><br><br><br>

### * Notice that both `Character m2 = m;` and `Character m3(m);` call the copy constructor. They are two different syntaxes for the same thing.

### * The default copy constructor that the compiler generates does exactly what our explicit one does -- it copies each member using that member's own copy constructor. For `int`, `string`, `vector`, etc., this produces a completely independent deep copy. 

### * But for **raw pointers** (`int*`, `char*`, etc.), copying just copies the address -- NOT the data it points to. This is where things can go wrong, and we'll see why in section 5.

<br><br><br><br><br>
<br><br><br><br><br>

<br><br><br><br><br>
<br><br><br><br><br>

In [ ]:
////copyYouTry.cpp
// Example 2d: YOU TRY -- predict the output!
// Draw a memory diagram showing both objects after all code executes.
#include<iostream>
#include<string>
class Character{
   public: std::string name_; int lives_, coins_;
   Character(std::string n){
        this->name_ = n; this->lives_ = 3; this->coins_ = 0;
   }
   void display() const {
        std::cout << this->name_ << ": " << this->lives_ << " lives, " << this->coins_ << " coins\n";
   }
   void collect_coins(int num_coins){
       this->coins_ += num_coins;
   }
   bool die(){
       this->lives_ -= 1;
       return this->lives_ > 0;
   }
};
int main(){
    Character m("Mario");
    m.collect_coins(50);
    m.die();
    Character m2 = m;
    m2.name_ = "Wario";
    m.collect_coins(30);
    m2.die();
    m2.die();
    std::cout << m2.die() << "\n";
    m.display();
    m2.display();
}


<br><br><br><br><br>
<br><br><br><br><br>


<br><br><br><br><br>
<br><br><br><br><br>

# 3. `this` Is a Pointer

### * We've been writing `this->name_`, `this->coins_`, etc. inside methods. But what exactly *is* `this`?

### * `this` is a **pointer**. Inside any method, `this` holds the address of the object the method was called on.

### * Let's prove it.

<br><br><br><br><br>

In [ ]:
////thisIsAPointer.cpp
// Example 3a: this is a pointer!
#include<iostream>
#include<string>
class Character{
   public: std::string name_; int lives_, coins_;
   Character(std::string n){
        this->name_ = n; this->lives_ = 3; this->coins_ = 0;
   }
   void display() const {
        std::cout << this->name_ << ": " << this->lives_ << " lives, " << this->coins_ << " coins\n";
   }
   void whoAmI() const {
        // Print name and address (this)

   }
};
int main(){
    Character m("Mario");
    Character l("Luigi");

    // Call whoAmI on both and compare to &m, &l


}

<br><br><br><br><br>

### * So `this->name_` literally means: "I am a pointer (`this`), go to the object I point to (`->`), and access its `name_` member." That's the arrow operator — the same `->` we'll use when we have our own pointers to objects on the heap.

### * You *could* write `(*this).name_` instead of `this->name_` — same thing. The arrow is just shorthand.

<br><br><br><br><br>

In [ ]:
////pointersObjects.cpp
// Example 3b: Pointers to objects
#include<iostream>
#include<string>
class Character{
   public: std::string name_; int lives_, coins_;
   Character(std::string n){
        this->name_ = n; this->lives_ = 3; this->coins_ = 0;
   }
   void display() const {
        std::cout << this->name_ << ": " << this->lives_ << " lives, " << this->coins_ << " coins\n";
   }
   void collect_coins(int num_coins){
       this->coins_ += num_coins;
   }
};
int main(){
    Character m("Mario");
    Character m2 = m;
    std::cout << (&m == &m2) << "\n";
    std::cout << &m << " " << &m2 << "\n";
    m2.collect_coins(20);
    m.display();
    m2.display();

    Character* ptr = &m;  // ptr POINTS to m -- no copy, same object
    ptr->collect_coins(100);
    m.display();  // m now has 100 coins!
}



### * With the pointer, changes through `ptr` affect `m` because they refer to the same object. With the copy, `m2` is independent.

<br><br><br><br><br>
<br><br><br><br><br>

# 4. Dynamic Memory Allocation of Objects

### * Now that we know `this` is a pointer and `->` means "follow the pointer and access a member," let's use that with our own pointers.

### * We've already seen we can allocate memory on the heap using `new` (and deallocate using `delete`). We can do this with objects of our own classes too.

### * Notice that in this version of the `Employee` class, we've made the attributes `private` -- this enforces encapsulation! Client code can only interact with Employee objects through the public methods.

<br><br><br><br><br>
<br><br><br><br><br>

In [ ]:
////employeeDynamicMemory.cpp
// Example 4a: Dynamic Memory allocation of objects
#include<iostream>
#include<string>
class Employee{
    private:
        std::string name_; std::string role_; double salary_; 
    public:
    Employee(std::string n, std::string r){
        this->name_ = n;
        this->role_ = r;
        this->salary_ = 10000;
    }
    void giveRaise(double percent){
        this->salary_ *= (1 + percent/100);
    }
    void display() const { // const is used for accessor methods
        std::cout << this->name_ << " has role " << this->role_ << " and salary " << this->salary_ << "\n";
    }
    ~Employee
        std::cout << "Calling the destructor for " 
};
int main(){
    // Allocate an Employee on the heap using new
    Employee* ptrE = new Employee("Simon", "amazin");
    
    //ptrE.display(); // this won't work -- e is a pointer, not an Employee
    // Dereference first, then call the method
    (*ptrE).display();

    
    // Arrow operator shorthand
    ptrE-> display();
    // same thing for attributes: e->role_, not e.role_
    // Give a raise and display
    ptrE->giveRaise(15);
    ptrE-> display();
        
    // Free the memory
    delete ptrE;
    ptrE = nullptr;
}

### * When we use `new`, the object is created on the **heap** instead of the stack. The variable `e` is a pointer that lives on the stack but points to the object on the heap.

### * Since `e` is a pointer, we can't use `e.display()` -- we need to dereference first: `(*e).display()`. The **arrow operator** `e->display()` is shorthand for this.

### * And since we used `new`, we are responsible for calling `delete` to free the memory when we're done!

<br><br><br><br><br>
<br><br><br><br><br>



### * What happens if we forget to `delete`? The object stays on the heap — **memory leak**.

### * The **destructor** (`~ClassName()`) is a special method called automatically when:
- A stack object goes out of scope
- `delete` is called on a heap object

### * Let's add one to see when cleanup happens.

<br><br><br><br><br>

<br><br><br><br><br>

### * **Important:** you do NOT `delete` the object itself inside the destructor. The destructor cleans up resources the object *owns* — then the object itself is automatically reclaimed. Trying to `delete this` or `delete &(this->name_)` would crash.

### * **When do you need a custom destructor?** For `Employee`, we don't! The default destructor handles `string`, `int`, `double`, etc. You only need one when your class has **raw pointer members** pointing to heap memory you allocated with `new`.

### * *Side note:* `std::unique_ptr` can manage heap memory automatically — but we'll stick with raw pointers for now so you understand what's happening under the hood.

<br><br><br><br><br>
<br><br><br><br><br>

<br><br><br><br><br>
<br><br><br><br><br>

In [ ]:
////characterHeap.cpp
// Example 4b: YOU TRY -- Character on the heap
// Create a Character on the heap, give it 100 coins using the arrow operator,
// display the character, and then free the memory.
#include<iostream>
#include<string>
class Character{
   public: std::string name_; int lives_, coins_;
   Character(std::string n){
        this->name_ = n; this->lives_ = 3; this->coins_ = 0;
   }
   void display() const {
        std::cout << this->name_ << ": " << this->lives_ << " lives, " << this->coins_ << " coins\n";
   }
   void collect_coins(int num_coins){
       this->coins_ += num_coins;
   }
   bool die(){
       this->lives_ -= 1;
       return this->lives_ > 0;
   }
};
int main(){
    // Create a Character on the heap, give it 100 coins, display it, and free the memory




}

<br><br><br><br><br>
<br><br><br><br><br>
<br><br><br><br><br>
<br><br><br><br><br>

# 5. When Default Copying Goes Wrong

### * Right now, our `Character` class tracks the *current* coin count. But what if we wanted to track a **history** of coin pickups — every time the character collects coins, we log the amount?

### * We could use a **heap-allocated array** for this. Let's say we fix the max size at 10 entries (in practice we'd want something more flexible, but this keeps things simple for now).



In [ ]:
////history.cpp
// Example 5a: Character with coin history
// YOU TRY: What's wrong with collect_coins? We want to log each pickup
// in the next available slot -- but how do we know which slot that is?
// What member variable should we add?
#include<iostream>
#include<string>
class Character{
   std::string name_;
   int lives_, coins_;
   int* history_;  // heap-allocated array to track coin pickups

   public:
   Character(std::string n){
        this->name_ = n; this->lives_ = 3; this->coins_ = 0;
        this->history_ = new int[10]; // allocate space for 10 entries
        for(int i = 0; i < 10; i++) this->history_[i] = 0; // initialize to zeros
   }
   void collect_coins(int num_coins){
       this->coins_ += num_coins;
       // log it in history... but which slot?
       if( i < 10)
       this->history_[???] = num_coins;
   }
   void display() const {
        std::cout << this->name_ << ": " << this->lives_ << " lives, " << this->coins_ << " coins\n";
   }
   ~Character(){
        delete[] this->history_;
   }
};

<br><br><br><br><br>
<br><br><br><br><br>

<br><br><br><br><br>

### * Great — now what happens when we **copy** a Character?

### * Remember: the default copy constructor copies each member one by one. `name_` is a `string` (independent copy), `coins_` is an `int` (independent copy), `history_count_` is an `int` (independent copy)... but `history_` is a **pointer**. What does copying a pointer do?

### * Let's find out.

<br><br><br><br><br>

In [ ]:
////shallowCopyBug.cpp
// Example 5b: Copying a Character -- what goes wrong?
#include<iostream>
#include<string>
class Character{
   public:
   std::string name_;
   int lives_, coins_;
   int* history_;
   int history_count_;

   Character(std::string n){
        this->name_ = n; this->lives_ = 3; this->coins_ = 0;
        this->history_ = new int[10];
        for(int i = 0; i < 10; i++) this->history_[i] = 0;
        this->history_count_ = 0;
   }
   void collect_coins(int num_coins){
       this->coins_ += num_coins;
       this->history_[this->history_count_] = num_coins;
       this->history_count_++;
   }
   void show_history() const {
       std::cout << this->name_ << "'s coin history: ";
       for(int i = 0; i < this->history_count_; i++){
           std::cout << this->history_[i] << " ";
       }
       std::cout << "\n";
   }
   void display() const {
        std::cout << this->name_ << ": " << this->lives_ << " lives, " << this->coins_ << " coins\n";
   }
   ~Character(){
        std::cout << "Destructor for " << this->name_ << ", history_ at " << this->history_ << "\n";
        delete[] this->history_;
   }
};
int main(){
    Character m("Mario");
    m.collect_coins(10);
    m.collect_coins(25);

    Character m2 = m;  // default copy
    m2.name_ = "Wario";
    m2.collect_coins(99);

    // Do they share the same history_ pointer?
    std::cout << "m.history_ address:  " << m.history_ << "\n";
    std::cout << "m2.history_ address: " << m2.history_ << "\n";

    m.show_history();
    m2.show_history();  // uh oh...
    // when main ends, both destructors delete[] the SAME memory -- crash!

}

// You try (comment out the main above and uncomment the one below)
// int main(){
//      Character m("Mario");
//      m.collect_coins(10);
//      m.collect_coins(25);
//      Character m2 = m;
//      m2.collect_coins(99);
//      m.show_history();
//      m2.show_history();
// }

// Which values are shared between m and m2? Which are independent? 
// Hint: coins_ and history_count_ are ints, but history_ is a pointer.)

wrote file shallowCopyBug.cpp
$> g++   shallowCopyBug.cpp    -o shallowCopyBug
$> ./shallowCopyBug 
m.history_ address:  0x592b3cd102b0
m2.history_ address: 0x592b3cd102b0
Mario's coin history: 10 25 
Wario's coin history: 10 25 99 
Destructor for Wario, history_ at 0x592b3cd102b0
Destructor for Mario, history_ at 0x592b3cd102b0


free(): double free detected in tcache 2
Aborted (core dumped)
executable failed with exit code 134


<br><br><br><br><br>

### * **What went wrong?** `history_` is a pointer — so the default copy just copied the *address*. Both `m` and `m2` share the **same** array on the heap. This is a **shallow copy**.

### * Two problems:
1. Wario's `collect_coins(99)` wrote into the shared array — so Mario's history is corrupted too
2. Both destructors try to `delete[]` the same memory — **double free** crash

### * **The fix:** write a custom copy constructor that allocates *new* memory and copies the *data* — a **deep copy**.

<br><br><br><br><br>

In [ ]:
////deepCopyFix.cpp
// Example 5c: The fix -- custom copy constructor (deep copy)
#include<iostream>
#include<string>
class Character{
   public:
   std::string name_;
   int lives_, coins_;
   int* history_;
   int history_count_;

   Character(std::string n){
        this->name_ = n; this->lives_ = 3; this->coins_ = 0;
        this->history_ = new int[10];
        for(int i = 0; i < 10; i++) this->history_[i] = 0;
        this->history_count_ = 0;
   }
   // Custom copy constructor -- deep copy!
   Character(const Character& other){
        // TODO: copy the non-pointer members as usual
        // Then allocate NEW memory for history_ and copy the DATA



   }
   void collect_coins(int num_coins){
       this->coins_ += num_coins;
       this->history_[this->history_count_] = num_coins;
       this->history_count_++;
   }
   void show_history() const {
       std::cout << this->name_ << "'s coin history: ";
       for(int i = 0; i < this->history_count_; i++){
           std::cout << this->history_[i] << " ";
       }
       std::cout << "\n";
   }
   void display() const {
        std::cout << this->name_ << ": " << this->lives_ << " lives, " << this->coins_ << " coins\n";
   }
   ~Character(){
        std::cout << "Destructor for " << this->name_ << ", history_ at " << this->history_ << "\n";
        delete[] this->history_;
   }
};
int main(){
    Character m("Mario");
    m.collect_coins(10);
    m.collect_coins(25);

    Character m2 = m;  // now calls OUR copy constructor
    m2.name_ = "Wario";
    m2.collect_coins(99);

    // Different addresses now!
    std::cout << "m.history_ address:  " << m.history_ << "\n";
    std::cout << "m2.history_ address: " << m2.history_ << "\n";

    m.show_history();   // Mario's history is safe!
    m2.show_history();
}

<br><br><br><br><br>

### * Now each object has its own separate copy of the `history_` array. No shared data, no double free.

### * **Rule of thumb:** if your class needs a custom destructor (because it `delete`s a pointer member), you almost certainly also need a custom copy constructor (and a custom copy assignment `operator=`). This is the **Rule of Three** — we'll formalize it next time.

<br><br><br><br><br>



<br><br><br><br><br>
<br><br><br><br><br>

# 6. A Brief Intro to References and `const`

### * We promised to circle back to `&` in `const Character& other` and `const` after method signatures.

<br><br><br><br><br>

## References (`&`)

### * A **reference** is an alias — another name for an existing variable. No copy is made.

### * Unlike a pointer: can't be null, can't be reassigned, no `*` needed.

<br><br><br><br><br>

In [ ]:
////references.cpp
// Example 6a: References as aliases
#include<iostream>
int main(){
    int x = 5;
    int& r = x;  // r IS x -- same object, different name
    r = 99;
    std::cout << "x = " << x << "\n";  // what does this print?
    std::cout << "r = " << r << "\n";
    std::cout << "&x = " << &x << ", &r = " << &r << "\n";  // same address!
}

<br><br><br><br><br>

### * Most common use: **function parameters**. Pass by reference = work with the original, no copy.

### * That's why the copy constructor takes `const Character& other` — receives the original directly (no copy → no infinite loop).

<br><br><br><br><br>

In [ ]:
////passbyref.cpp
// Example 6b: Pass by reference
#include<iostream>
void addTen(int& n){  // n is a reference to whatever was passed in
    n += 10;          // modifies the original!
}
int main(){
    int x = 5;
    addTen(x);
    std::cout << "x = " << x << "\n";  // x is now 15

    // Compare: pass by value (copy)
    // void addTen(int n){ n += 10; }  // this would NOT change x
}


<br><br><br><br><br><br><br><br><br><br>
## `const`

### * `const` = "this cannot be changed." Two uses today:

### **1. `const` parameters** — `const Character& other`:
### * `&` = pass by reference (no copy). `const` = promise not to modify. Together: look at the original safely.

<br><br><br><br><br>

### **2. `const` methods** — `void display() const`:
### * Means "this method won't modify the object." Can read members but not change them. Good practice for accessor/display methods.

<br><br><br><br><br>

In [ ]:
////constDemo.cpp
// Example 6c: const methods
#include<iostream>
#include<string>
class Character{
   public: std::string name_; int lives_, coins_;
   Character(std::string n){
        this->name_ = n; this->lives_ = 3; this->coins_ = 0;
   }
   void display() const {  // const = won't modify the object
        std::cout << this->name_ << ": " << this->lives_ << " lives, " << this->coins_ << " coins\n";
   }
   void collect_coins(int num_coins){  // NOT const -- it changes coins_
       this->coins_ += num_coins;
   }
};
int main(){
    const Character m("Mario");  // m is a const object
    m.display();       // OK -- display() is const
    // m.collect_coins(10);  // ERROR -- can't call non-const method on const object
}

### * We'll see `const` and references more throughout the semester — this is just a first look to demystify the syntax.

<br><br><br><br><br>
<br><br><br><br><br>